# movx YOLO26 pose fine-tuning

Full-parameter fine-tuning for `yolo26x-pose.pt` and `yolo26n-pose.pt` on the human-corrected bowling-arm data. The generated labels retain the standard 17-keypoint COCO head required by the app and TCN.

Use a Colab **T4 GPU**. Run the notebook top to bottom, leaving the final test and export switches off until validation-based model selection is complete.

## 1. Install the reproducible environment

The version is pinned to the version recorded in the currently deployed TFLite metadata. Restart the runtime only if Colab requests it.

In [ ]:
%pip install -q ultralytics==8.4.175 ai-edge-litert==2.2.0


In [ ]:
import csv
import gc
import json
import math
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda, '| PyTorch:', torch.__version__)

def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 2. Configuration

`EVENT_DATASET` is used for YOLO26x. `NANO_DATASET` additionally samples train-only context frames because nano processes complete clips. Checkpoints and final artifacts are written to Drive.

In [ ]:
REPO_URL = 'https://github.com/talhashafi-04/movx.git'  # Change if the repository is private/forked.
BRANCH = 'main'
REPO = Path('/content/movx')
EVENT_DATASET = Path('/content/movx_pose_event')
NANO_DATASET = Path('/content/movx_pose_nano')
DRIVE_ROOT = Path('/content/drive/MyDrive/movx_yolo26_finetune')
SEED = 42
IMGSZ = 640
EPOCHS_X = 80
EPOCHS_N = 100
PATIENCE = 18
CONTEXT_STRIDE = 5
RUN_BASELINES = True
RUN_SMOKE_TEST = True
TRAIN_X = True
TRAIN_N = True
RESUME = True
CHECKPOINT_NME_TOLERANCE = 0.02  # Treat NME values within 2% of the minimum as tied, then prefer angle accuracy.
RUN_FINAL_TEST = False  # Turn on once, after selecting checkpoints from validation.
EXPORT_MODELS = False   # Turn on only after validation is satisfactory.
RUN_TFLITE_VALIDATION = False  # Slow CPU check of the exported binaries.


## 3. Mount Drive and obtain the repository

The repository contains about 1.8 GB of trimmed videos, so the initial shallow clone can take a few minutes. For a private repository, authenticate Git before running this cell; do not paste a token into the notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)

commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', commit)


## 4. Audit and build datasets

The builder prefers `_corrected.zip`, falls back to the corrected COCO JSON, extracts frames from the videos, uses full-person teacher boxes, and replaces COCO right shoulder/elbow/wrist indices `6, 8, 10` with human coordinates. Validation and test never receive pseudo-only context frames.

In [ ]:
builder = REPO / 'src/build_yolo_pose_dataset.py'
subprocess.run([sys.executable, '-u', str(builder), '--repo-root', str(REPO), '--audit-only'], check=True)

def build_if_incomplete(output, extra_args=()):
    required = (output / 'dataset.yaml', output / 'manifest.csv', output / 'summary.json')
    if all(path.is_file() for path in required):
        summary = json.loads((output / 'summary.json').read_text())
        print(f'Skipping completed dataset {output} ({summary["images"]} images)')
        return
    subprocess.run([
        sys.executable, '-u', str(builder), '--repo-root', str(REPO),
        '--output', str(output), '--teacher', 'average',
        '--pseudo-confidence', '0.25', '--overwrite', *extra_args,
    ], check=True)

build_if_incomplete(EVENT_DATASET)
build_if_incomplete(NANO_DATASET, ('--context-stride', str(CONTEXT_STRIDE)))


In [ ]:
event_summary = json.loads((EVENT_DATASET / 'summary.json').read_text())
nano_summary = json.loads((NANO_DATASET / 'summary.json').read_text())
assert event_summary['clips'] == 84
assert event_summary['images'] == 2378
assert event_summary['split_images'] == {'test': 473, 'train': 1566, 'val': 339}
assert nano_summary['split_images']['val'] == 339
assert nano_summary['split_images']['test'] == 473
assert nano_summary['split_images']['train'] > 1566
print('Event dataset:', json.dumps(event_summary, indent=2))
print('Nano dataset:', json.dumps(nano_summary, indent=2))


## 5. Visual label sanity check

Inspect several samples before spending GPU time. The full-person box is white, all visible keypoints are green, and the human-corrected right arm is red.

In [ ]:
ARM = [6, 8, 10]
EDGES = [(5,7),(7,9),(6,8),(8,10),(5,6),(5,11),(6,12),(11,12),
         (11,13),(13,15),(12,14),(14,16),(0,1),(0,2),(1,3),(2,4)]

def show_label(image_path):
    image_path = Path(image_path)
    label_path = Path(str(image_path).replace('/images/', '/labels/')).with_suffix('.txt')
    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    values = np.array([float(v) for v in label_path.read_text().split()])
    box = values[1:5] * [w, h, w, h]
    cx, cy, bw, bh = box
    points = values[5:].reshape(17, 3)
    points[:, :2] *= [w, h]
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(image)
    ax.add_patch(plt.Rectangle((cx-bw/2, cy-bh/2), bw, bh, fill=False, color='white', lw=2))
    for a, b in EDGES:
        if points[a, 2] and points[b, 2]:
            ax.plot(points[[a,b], 0], points[[a,b], 1], color='lime', lw=1)
    for index, (x, y, visible) in enumerate(points):
        if visible:
            ax.scatter(x, y, color='red' if index in ARM else 'lime', s=35)
    ax.set_title(image_path.name)
    ax.axis('off')
    plt.show()

samples = sorted((EVENT_DATASET / 'images/val').glob('*.jpg'))
for sample in samples[::max(1, len(samples)//3)][:3]:
    show_label(sample)


## 6. Baseline validation

Record pretrained validation metrics before training. These standard metrics include pseudo-labelled joints; the human-arm evaluator below is the primary selection metric. Do not inspect the test split yet.

In [ ]:
if RUN_BASELINES:
    for weights, batch in [('yolo26n-pose.pt', 8), ('yolo26x-pose.pt', 1)]:
        print('Baseline:', weights)
        clear_cuda()
        baseline_model = YOLO(weights)
        try:
            baseline_model.val(
                data=str(EVENT_DATASET / 'dataset.yaml'), split='val', imgsz=IMGSZ,
                batch=batch, device=0, plots=True, project=str(DRIVE_ROOT / 'baseline'),
                name=Path(weights).stem, seed=SEED,
            )
        finally:
            del baseline_model
            clear_cuda()


## 7. Full-parameter training helpers

No layers are frozen. A small AdamW learning rate, AMP, conservative geometry, no mosaic, and no horizontal flip are used. `batch=0.80` lets Ultralytics profile roughly 80% of T4 memory. Existing `last.pt` checkpoints are resumed automatically.

In [ ]:
RUNS = DRIVE_ROOT / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)

def train_pose(weights, dataset_yaml, run_name, epochs, lr0):
    run_dir = RUNS / run_name
    last = run_dir / 'weights/last.pt'
    if RESUME and last.is_file():
        print('Resuming', last)
        return YOLO(str(last)).train(resume=True)
    model = YOLO(weights)
    return model.train(
        data=str(dataset_yaml), project=str(RUNS), name=run_name, exist_ok=True,
        epochs=epochs, patience=PATIENCE, imgsz=IMGSZ, batch=0.80, device=0,
        optimizer='AdamW', lr0=lr0, lrf=0.05, cos_lr=True, warmup_epochs=3,
        amp=True, workers=2, cache='disk', seed=SEED, deterministic=True,
        fliplr=0.0, mosaic=0.0, degrees=2.0, translate=0.05, scale=0.20,
        perspective=0.0, shear=0.0, hsv_h=0.01, hsv_s=0.25, hsv_v=0.20,
        save=True, save_period=5, plots=True, verbose=True,
    )


## 8. Five-epoch nano smoke test

This catches bad labels, OOM settings, and dependency problems cheaply. Inspect loss curves and plotted validation batches before starting the long runs.

In [ ]:
if RUN_SMOKE_TEST:
    previous_resume = RESUME
    RESUME = False
    train_pose('yolo26n-pose.pt', EVENT_DATASET / 'dataset.yaml', 'nano_smoke', 5, 3e-4)
    RESUME = previous_resume
    clear_cuda()


## 9. Train YOLO26x-pose

The accuracy model trains on human-corrected event windows only. On a T4 its automatically selected batch will be much smaller than nano's.

In [ ]:
if TRAIN_X:
    train_pose(
        'yolo26x-pose.pt', EVENT_DATASET / 'dataset.yaml',
        'yolo26x_pose_movx', EPOCHS_X, 1e-4,
    )
    clear_cuda()


## 10. Train YOLO26n-pose

Nano uses the context-enhanced training set, while its validation and test sets remain identical to the x dataset.

In [ ]:
if TRAIN_N:
    train_pose(
        'yolo26n-pose.pt', NANO_DATASET / 'dataset.yaml',
        'yolo26n_pose_movx', EPOCHS_N, 3e-4,
    )
    clear_cuda()


## 11. Human-arm evaluation

This evaluator ignores pseudo-labelled joints and scans `best.pt`, `last.pt`, and every saved `epoch*.pt`. It reports normalized error and PCK for the three human points, elbow-angle error, and missed-person rate. Selection first minimizes misses, treats NME values within 2% of the minimum as tied, and then prefers the lowest elbow-angle error. This prevents a negligible NME improvement from selecting a checkpoint with much worse bowling-arm geometry. Use validation for selection; enable the test split only once.

In [ ]:
# Standard COCO indices: right shoulder, right elbow, right wrist.
ARM = np.array([6, 8, 10], dtype=int)
SELECTION_POLICY_ID = 'nme_within_2pct_then_angle_v1'
RUNS = DRIVE_ROOT / 'runs'

def elbow_angle(points):
    a = points[0] - points[1]
    b = points[2] - points[1]
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return np.nan
    return np.degrees(np.arccos(np.clip(np.dot(a, b) / denom, -1, 1)))

def arm_metrics(weights, dataset_root, split='val', batch=8):
    dataset_root = Path(dataset_root)
    image_paths = sorted((dataset_root / 'images' / split).glob('*.jpg'))
    assert image_paths, f'No {split} images found below {dataset_root}'
    clear_cuda()
    model = YOLO(str(weights))
    inference_device = 'cpu' if str(weights).endswith('.tflite') else 0
    errors, angle_errors = [], []
    pck05_hits = pck10_hits = 0
    misses = 0
    try:
        predictions = model.predict(
            [str(path) for path in image_paths], imgsz=IMGSZ, batch=batch,
            device=inference_device, stream=True, verbose=False,
        )
        for result in predictions:
            image_path = Path(result.path)
            label_path = dataset_root / 'labels' / split / f'{image_path.stem}.txt'
            values = np.array([float(v) for v in label_path.read_text().split()])
            h, w = result.orig_shape
            gt_all = values[5:].reshape(17, 3)
            gt = gt_all[ARM, :2] * [w, h]
            bbox_diagonal = np.linalg.norm(values[3:5] * [w, h])
            if result.boxes is None or len(result.boxes) == 0 or result.keypoints is None:
                misses += 1
                continue
            boxes = result.boxes.xyxy.cpu().numpy()
            index = int(np.argmax((boxes[:, 2]-boxes[:, 0]) * (boxes[:, 3]-boxes[:, 1])))
            pred = result.keypoints.xy[index].cpu().numpy()[ARM]
            normalized = np.linalg.norm(pred - gt, axis=1) / bbox_diagonal
            errors.extend(normalized.tolist())
            pck05_hits += int(np.sum(normalized <= 0.05))
            pck10_hits += int(np.sum(normalized <= 0.10))
            angle_errors.append(abs(elbow_angle(pred) - elbow_angle(gt)))
    finally:
        predictions = None
        result = None
        model = None
        clear_cuda()
    errors = np.asarray(errors)
    report = {
        'model': str(weights), 'split': split, 'images': len(image_paths), 'misses': misses,
        'miss_rate': misses / max(1, len(image_paths)),
        'arm_nme_detected': float(np.mean(errors)) if len(errors) else None,
        'pck@0.05': pck05_hits / max(1, len(image_paths) * 3),
        'pck@0.10': pck10_hits / max(1, len(image_paths) * 3),
        'elbow_angle_mae_deg': float(np.nanmean(angle_errors)) if angle_errors else None,
    }
    print(json.dumps(report, indent=2))
    return report

def checkpoint_candidates(run_dir):
    weights_dir = Path(run_dir) / 'weights'
    candidates = [path for path in (weights_dir / 'best.pt', weights_dir / 'last.pt') if path.is_file()]
    epochs = sorted(
        weights_dir.glob('epoch*.pt'),
        key=lambda path: int(path.stem.removeprefix('epoch')),
    )
    candidates.extend(path for path in epochs if path not in candidates)
    if not candidates:
        raise FileNotFoundError(f'No checkpoints found in {weights_dir}')
    return candidates

def select_application_checkpoint(reports, nme_tolerance=CHECKPOINT_NME_TOLERANCE):
    # Missed people are always worse. Within the best miss count, NME differences
    # smaller than the tolerance are treated as noise and angle MAE breaks the tie.
    fewest_misses = min(report['misses'] for report in reports)
    detected = [
        report for report in reports
        if report['misses'] == fewest_misses and report['arm_nme_detected'] is not None
    ]
    if not detected:
        raise RuntimeError('Every checkpoint missed every validation image')
    minimum_nme = min(report['arm_nme_detected'] for report in detected)
    near_minimum = [
        report for report in detected
        if report['arm_nme_detected'] <= minimum_nme * (1 + nme_tolerance)
    ]
    return min(
        near_minimum,
        key=lambda report: (
            report['elbow_angle_mae_deg'] if report['elbow_angle_mae_deg'] is not None and math.isfinite(report['elbow_angle_mae_deg']) else math.inf,
            report['arm_nme_detected'],
            -report['pck@0.05'],
            {'last.pt': 0, 'best.pt': 1}.get(report['checkpoint'], 2),
            report['checkpoint'],
        ),
    )

run_configs = [
    {'run': 'yolo26n_pose_movx', 'batch': 8},
    {'run': 'yolo26x_pose_movx', 'batch': 1},
]
evaluation = {
    'selection_policy': {
        'id': SELECTION_POLICY_ID,
        'nme_relative_tolerance': CHECKPOINT_NME_TOLERANCE,
        'description': 'Minimize misses; keep checkpoints within 2% of minimum arm NME; then minimize elbow-angle MAE.',
    },
    'baselines': [],
    'checkpoint_comparisons': [],
}

if RUN_BASELINES:
    for weights, batch in [('yolo26n-pose.pt', 8), ('yolo26x-pose.pt', 1)]:
        baseline = arm_metrics(weights, EVENT_DATASET, 'val', batch)
        baseline['checkpoint'] = Path(weights).name
        evaluation['baselines'].append(baseline)

for config in run_configs:
    run_dir = RUNS / config['run']
    validation_reports = []
    for checkpoint in checkpoint_candidates(run_dir):
        print(f'Validating {config["run"]}: {checkpoint.name}')
        report = arm_metrics(checkpoint, EVENT_DATASET, 'val', config['batch'])
        report['checkpoint'] = checkpoint.name
        validation_reports.append(report)

    application_best = select_application_checkpoint(validation_reports)
    ultralytics_best = next(
        report for report in validation_reports if report['checkpoint'] == 'best.pt'
    )
    comparison = {
        'run': config['run'],
        'selection_split': 'val',
        'ultralytics_best': ultralytics_best,
        'application_best': application_best,
        'all_validation_checkpoints': validation_reports,
        'test_comparison': [],
    }

    if RUN_FINAL_TEST:
        test_paths = [Path(application_best['model'])]
        best_path = Path(ultralytics_best['model'])
        if best_path != test_paths[0]:
            test_paths.append(best_path)
        for checkpoint in test_paths:
            test_report = arm_metrics(checkpoint, EVENT_DATASET, 'test', config['batch'])
            test_report['checkpoint'] = checkpoint.name
            comparison['test_comparison'].append(test_report)

    evaluation['checkpoint_comparisons'].append(comparison)
    print(
        f'{config["run"]}: Ultralytics={ultralytics_best["checkpoint"]}, '
        f'application={application_best["checkpoint"]}'
    )

comparison_path = DRIVE_ROOT / 'arm_checkpoint_comparison.json'
comparison_path.write_text(json.dumps(evaluation, indent=2))
print('Saved checkpoint comparison to', comparison_path)


## 12. Export deployment-compatible TFLite models

The selected checkpoint paths are loaded from `arm_checkpoint_comparison.json`, so this cell also works after a runtime restart and cannot silently fall back to `best.pt`. Nano remains FP32. Extra-large is exported as dynamic `w8a32`: INT8 weights with FP32 activations, matching the existing app model. Both retain 640×640 input, raw output, batch 1, and 17×3 keypoints.

In [ ]:
ARTIFACTS = DRIVE_ROOT / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

def find_exported_tflite(export_result):
    path = Path(str(export_result))
    if path.is_file() and path.suffix == '.tflite':
        return path
    root = path if path.is_dir() else path.parent
    candidates = sorted(root.rglob('*.tflite'))
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one TFLite below {root}, found {candidates}')
    return candidates[0]

def selected_checkpoint(run_name):
    comparison_path = DRIVE_ROOT / 'arm_checkpoint_comparison.json'
    if not comparison_path.is_file():
        raise FileNotFoundError(f'Run the human-arm evaluation first: {comparison_path} is missing')
    evaluation = json.loads(comparison_path.read_text())
    policy = evaluation.get('selection_policy', {})
    if policy.get('id') != 'nme_within_2pct_then_angle_v1':
        raise RuntimeError('Checkpoint report uses the old selector; rerun the updated human-arm evaluation cell')
    comparison = next(
        item for item in evaluation['checkpoint_comparisons'] if item['run'] == run_name
    )
    checkpoint = Path(comparison['application_best']['model'])
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Selected checkpoint is missing: {checkpoint}')
    return checkpoint

def export_selected_tflite(weights, destination, quantize=None):
    clear_cuda()
    export_model = YOLO(str(weights))
    try:
        kwargs = {
            'format': 'tflite', 'imgsz': IMGSZ, 'batch': 1, 'nms': False,
        }
        if quantize is not None:
            kwargs['quantize'] = quantize
        export_result = export_model.export(**kwargs)
        shutil.copy2(find_exported_tflite(export_result), destination)
    finally:
        export_model = None
        clear_cuda()
    return destination

if EXPORT_MODELS:
    N_EXPORT_WEIGHTS = selected_checkpoint('yolo26n_pose_movx')
    X_EXPORT_WEIGHTS = selected_checkpoint('yolo26x_pose_movx')
    print('Nano export checkpoint:', N_EXPORT_WEIGHTS)
    print('X export checkpoint:', X_EXPORT_WEIGHTS)

    n_artifact = export_selected_tflite(
        N_EXPORT_WEIGHTS, ARTIFACTS / 'yolo26n-pose.tflite',
    )
    x_artifact = export_selected_tflite(
        X_EXPORT_WEIGHTS, ARTIFACTS / 'yolo26x-pose_int8w.tflite', quantize='w8a32',
    )
    export_manifest = {
        'yolo26n': {'checkpoint': str(N_EXPORT_WEIGHTS), 'artifact': str(n_artifact)},
        'yolo26x': {'checkpoint': str(X_EXPORT_WEIGHTS), 'artifact': str(x_artifact), 'quantize': 'w8a32'},
    }
    (ARTIFACTS / 'export_manifest.json').write_text(json.dumps(export_manifest, indent=2))
    print('Exported deployment artifacts to', ARTIFACTS)


## 13. Verify metadata and tensor contract

Do not replace the app models unless these assertions pass and exported validation remains acceptable.

In [ ]:
from ai_edge_litert.interpreter import Interpreter

def verify_tflite(path, expected_quantize):
    path = Path(path)
    with zipfile.ZipFile(path) as archive:
        metadata = json.loads(archive.read('metadata.json'))
    assert metadata['imgsz'] == [640, 640]
    assert metadata['kpt_shape'] == [17, 3]
    assert metadata['end2end'] is False
    assert metadata['args']['quantize'] == expected_quantize
    interpreter = Interpreter(str(path))
    interpreter.allocate_tensors()
    inputs = interpreter.get_input_details()
    outputs = interpreter.get_output_details()
    assert list(inputs[0]['shape']) == [1, 3, 640, 640]
    assert 56 in list(outputs[0]['shape']), outputs[0]['shape']
    report = {
        'path': str(path), 'bytes': path.stat().st_size, 'metadata': metadata,
        'input_shape': inputs[0]['shape'].tolist(), 'input_dtype': str(inputs[0]['dtype']),
        'output_shape': outputs[0]['shape'].tolist(), 'output_dtype': str(outputs[0]['dtype']),
    }
    print(json.dumps(report, indent=2))
    return report

if EXPORT_MODELS:
    deployment_reports = [
        verify_tflite(ARTIFACTS / 'yolo26n-pose.tflite', None),
        verify_tflite(ARTIFACTS / 'yolo26x-pose_int8w.tflite', 'w8a32'),
    ]
    (ARTIFACTS / 'verification.json').write_text(json.dumps(deployment_reports, indent=2))


## 14. Validate the exported binaries

LiteRT validation is CPU-bound in Colab and can be slow, so it is separately guarded. This step measures the actual deployment artifacts rather than assuming the PyTorch result survives export.

In [ ]:
if RUN_TFLITE_VALIDATION:
    assert EXPORT_MODELS, 'Export the models first'
    tflite_reports = [
        arm_metrics(ARTIFACTS / 'yolo26n-pose.tflite', EVENT_DATASET, 'val', 1),
        arm_metrics(ARTIFACTS / 'yolo26x-pose_int8w.tflite', EVENT_DATASET, 'val', 1),
    ]
    (ARTIFACTS / 'tflite_arm_metrics.json').write_text(json.dumps(tflite_reports, indent=2))


## 15. Final integration checks

Before replacing `movx_dev_server/models/`:

1. Run validation on the exported TFLite models, not only `best.pt`.
2. Run the complete two-pass pipeline on held-out p6/p10 clips.
3. Compare AH/BR frame error, rejection rate, latency, and arm-angle error with production.
4. Reevaluate the TCN because its 97 input features depend on pose coordinates and confidence. If event accuracy shifts, regenerate pose sequences and retrain the TCN on outputs from both tuned models.
5. Keep the previous TFLite files as rollback artifacts; replace production binaries only in a separate reviewed commit.